In [ ]:
import pandas as pd
import numpy as np
import os
import mo_gymnasium as mo_gym
import gymnasium as gym
from env.user_sim import UserSimEnv
import stable_baselines3 as sb3
import torch
from morl_baselines.multi_policy.morld import morld
import scipy.stats as stats
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from ipywidgets import interact
import src.old.value_iteration as value_iteration
import utils
import process_data as dp
import mdp_utils

User State Index: 5
Count State Index: 255
Original: (1, 2, 3, 3, 3, 3)
Reconstructed: (1, 2, 3, 3, 3, 3)


In [2]:
cluster_col = 'cluster_all'
state_features = ['tiredness', 'time_avail', 'pu_state']
feature_names = ['Tiredness', 'Time available', 'Perceived usefulness']
cluster_vars = ['obs_diff', 'obs_time', 'obs_liked', 'obs_pu']
cluster_cols = [col + '_cluster' for col in cluster_vars]

reward_cols = ["r_time", "obs_liked", "obs_pu", "r_expert", "r_diversity"]
obj_names = ["Time Required", "Fun", "Perceived Usefulness", "Expert Usefulness", "Novelty"]

NUM_FEATURES = len(state_features)
NUM_VALS_PER_FEATURE = [2,3,2]
NUM_USER_STATES = np.prod(NUM_VALS_PER_FEATURE)
NUM_CLUSTERS = 6
MAX_COUNT = 2
NUM_COUNT_STATES = (MAX_COUNT+1)**NUM_CLUSTERS
NUM_STATES = NUM_USER_STATES * NUM_COUNT_STATES 
NUM_ACTIONS = NUM_CLUSTERS
NUM_OBJECTIVES = len(reward_cols)

data_folder = 'C:\\Users\\shirl\\Documents\\Studie\\2025-2026\\Thesis\\personalized-coping-challenges\\data\\functions\\3'
results_folder = 'C:\\Users\\shirl\\Documents\\Studie\\2025-2026\\Thesis\\personalized-coping-challenges\\results\\'

# Load data
action_data = pd.read_csv(os.path.join(data_folder, 'action_data.csv'))
samples = pd.read_csv(os.path.join(data_folder, 'samples.csv'))

actions_clustered, _, _ = dp.cluster_actions(action_data, cluster_vars, num_clusters=6)

df, initial_distribution = dp.process_samples(samples, actions_clustered, state_features, NUM_VALS_PER_FEATURE, cluster_col, MAX_COUNT, NUM_CLUSTERS)

Time rating percentiles:
33.333333333333336th percentile: 3.0000
66.66666666666667th percentile: 5.0000
100.0th percentile: 7.0000


In [3]:
P_comp = mdp_utils.compute_completion_probabilities_clustered(df, cluster_col, NUM_USER_STATES, NUM_ACTIONS, num_clusters=NUM_CLUSTERS, use_clusters=True)
R = mdp_utils.compute_rewards(df, P_comp, NUM_USER_STATES, NUM_COUNT_STATES, NUM_ACTIONS, NUM_OBJECTIVES, reward_cols, cluster_col)
P = mdp_utils.compute_transition_probabilities_clustered(df, NUM_USER_STATES, NUM_ACTIONS, cluster_col, num_clusters=NUM_CLUSTERS, use_clusters=True)

print("P shape:", P.shape)  # Should be (nU, nA, nU)
print("R shape:", R.shape)  # Should be (nU, nC, nA, nO)

df = df.merge(actions_clustered[['action_id'] + cluster_cols], on='action_id', how='left')

# Use overall clusters or specific reward clusters?
R_probs_diff = mdp_utils.compute_reward_probabilities(df, 'obs_diff', NUM_USER_STATES, NUM_ACTIONS, cluster_col)
R_probs_fun = mdp_utils.compute_reward_probabilities(df, 'obs_liked', NUM_USER_STATES, NUM_ACTIONS, cluster_col)
R_probs_pu = mdp_utils.compute_reward_probabilities(df, 'obs_pu', NUM_USER_STATES, NUM_ACTIONS, cluster_col)
R_probs_time = mdp_utils.compute_reward_probabilities(df, 'obs_time', NUM_USER_STATES, NUM_ACTIONS, cluster_col)

R_probs = {'obs_diff': R_probs_diff, 'obs_liked': R_probs_fun, 'obs_pu': R_probs_pu, 'obs_time': R_probs_time}

action_categories = actions_clustered[cluster_col].values  # cluster ID for each action (is same as action number if actions are clusters)
challenges_per_cluster = [[i for i in range(len(actions_clustered)) if actions_clustered[cluster_col].iloc[i] == cluster_id] for cluster_id in range(NUM_CLUSTERS)]
challenge_info = actions_clustered.set_index('action_id').to_dict('records')
expert_score_cols = ['score_acceptance', 'score_distraction', 'score_problem_solving', 'score_social_support']
expert_score_matrix = actions_clustered[expert_score_cols].values

P shape: (12, 6, 12)
R shape: (12, 729, 6, 5)


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
env = mo_gym.make('user_env', num_actions=NUM_CLUSTERS, 
                 num_objectives = NUM_OBJECTIVES, 
                 num_vals_per_feature=NUM_VALS_PER_FEATURE,
                 challenge_info=challenge_info, 
                 expert_ratings=expert_score_matrix,
                 challenges_per_cluster=challenges_per_cluster,
                 num_clusters=NUM_CLUSTERS,
                 transition_probs=P, 
                 completion_probs=P_comp,
                 reward_matrix=R, 
                 reward_probs=R_probs,
                 action_categories=action_categories,
                 initial_distribution=initial_distribution,
                 max_count=MAX_COUNT)

In [6]:
# lin_env = mo_gym.wrappers.LinearReward(env, weight=np.array([0.9, 0.1, 0.0]))

# # Run DQN agent!
# agent = sb3.DQN("MultiInputPolicy", lin_env)
# agent.learn(500)

In [7]:
num_evals = 30
pop_size = 10
num_steps_per_episode=28
total_steps = 500

model_name = f'morld_T={total_steps}_P={pop_size}_nO={NUM_OBJECTIVES}_nA={NUM_ACTIONS}_n_eval={num_evals}'
filename = model_name
model_file = os.path.join(results_folder, model_name, 'weights', filename)

morld_agent = morld.MORLD(env, policy_name="MOSACDiscrete", log=False, device=device, weight_init_method="uniform", weight_adaptation_method="PSA", pop_size=pop_size, exchange_every=100)
if os.path.exists(model_file + '.tar'):
    print("Loading pretrained MORLD model...")
    morld_agent.load(path=model_file + '.tar')

else:
    print("No pretrained MORLD model found. Training a new model...")
    eval_env = mo_gym.make('user_env', num_actions=NUM_CLUSTERS, 
                 num_objectives = NUM_OBJECTIVES, 
                 num_vals_per_feature=NUM_VALS_PER_FEATURE,
                 challenge_info=challenge_info, 
                 expert_ratings=expert_score_matrix,
                 challenges_per_cluster=challenges_per_cluster,
                 num_clusters=NUM_CLUSTERS,
                 transition_probs=P, 
                 completion_probs=P_comp,
                 reward_matrix=R, 
                 reward_probs=R_probs,
                 action_categories=action_categories,
                 initial_distribution=initial_distribution,
                 max_count=MAX_COUNT)
    ref_point = np.array([5.0, 5.0, 5.0, 5.0, 5.0])  # reference point for hypervolume calculation
    morld_agent.train(total_timesteps=total_steps, num_eval_episodes_for_front=num_evals, eval_env=eval_env, ref_point=ref_point)
    os.makedirs(os.path.join(results_folder, model_name, 'weights'))
    morld_agent.save(filename=model_file)



Weights: [[0.         0.         0.         0.         1.        ]
 [0.         0.         0.         1.         0.        ]
 [0.         0.         0.25818011 0.345144   0.39667588]
 [0.         0.         1.         0.         0.        ]
 [0.         0.50105089 0.         0.49894911 0.        ]
 [0.         1.         0.         0.         0.        ]
 [0.18423711 0.39131162 0.42445126 0.         0.        ]
 [0.50539281 0.         0.         0.49460719 0.        ]
 [0.50997842 0.         0.         0.         0.49002158]
 [1.         0.         0.         0.         0.        ]]
Neighborhoods: [[8], [4], [8], [6], [6], [4], [4], [8], [2], [8]]
No pretrained MORLD model found. Training a new model...
Starting training...
Current pareto archive:
[array([ 71.3413586 ,  73.79663835, 117.35133364,  17.92418517,
        14.33281881]), array([ 71.50619713,  71.26236132, 115.69239374,  18.08008299,
        14.23283006]), array([ 70.61722905,  73.07082998, 117.18358511,  18.06313603,
      

### Run simulations

In [13]:
def build_user_row(user, t, action, obs, obs_next, rewards, info):
    return {
            'user': user,
            't': t,
            'action': action,
            'state': obs,
            'next_state': obs_next,
            'counts': obs[3:],
            'num_completed': info['num_completed'],
            'counts_per_category': info['counts_per_category'].copy(),
            'expert_competencies': info['expert_competencies'].copy(),
            'rewards': rewards
        }

def simulate(env, num_users, policy=None, policy_name=None, verbose=False, T=28, num_vals=3, max_count=3, seed=66):
    data_list = [] 
    for user in range(num_users):
        user_seed = seed + user
        t = 0
        obs, _ = env.reset(seed=user_seed)
        done = False
        while not done and t < T:
            state_idx = env.unwrapped.get_full_state_index(obs)
            action = policy.eval(obs) if policy is not None else env.action_space.sample()
            obs_next, rewards, terminated, truncated, info = env.step(action)
            if verbose:
                print(f"Action: {action}, Rewards: {rewards}, Info: {info}")
            user_row = build_user_row(user, t, action, obs, obs_next, rewards, info)
            data_list.append(user_row)

            done = terminated or truncated
            obs = obs_next
            t += 1
    simulation_results = pd.DataFrame(data_list)
    simulation_results['policy'] = policy_name if policy_name is not None else 'Random'
    return simulation_results

In [9]:
NUM_USERS = 500
NUM_TIMESTEPS = 28

objectives = ["Time Required", "Fun", "Perceived Usefulness", "Expert Usefulness", "Diversity"]

In [10]:
policies = morld_agent.archive.individuals
print("Number of policies:", len(policies))

Number of policies: 33


In [26]:
selected_policies = policies[:5]  # Select the first 5 policies for simulation

save_path = results_folder + model_name + '/simulations/'
save_path = os.path.join(results_folder, model_name, 'simulations')

all_sim_results =[]
if os.path.exists(save_path):
    for i in range(len(selected_policies)):
        file_path = os.path.join(save_path, f'simulation_{i}.pkl')
        df = pd.read_pickle(file_path)
        all_sim_results.append(df)
else:
    os.makedirs(save_path)
    for i, policy in enumerate(selected_policies):
        print(f"Simulating Policy {i}:")
        print(f"  Weights: {policy.weights}")
        simulation_results = simulate(env, NUM_USERS, policy=policy.wrapped, policy_name=f'Policy_{i}')
        all_sim_results.append(simulation_results)
        save_file = f'simulation_{i}.pkl'
        save_to = os.path.join(save_path, save_file)
        simulation_results.to_pickle(save_to)

In [ ]:
random_sim_file = f'random/simulation_random_n={NUM_USERS}.pkl'
random_sim_path = os.path.join(results_folder, random_sim_file)

if os.path.exists(random_sim_path):
    simulation_random = pd.read_pickle(random_sim_path)
else:
    simulation_random = simulate(env, NUM_USERS, policy=None, policy_name='Random')
    simulation_random.to_pickle(random_sim_path)

In [28]:
import env.simulate as sim_utils
all_sim_results.append(simulation_random)
sim_utils.interactive_plot_objective(all_sim_results, objectives)

interactive(children=(Dropdown(description='Objective:', options=(('Time Required', 0), ('Fun', 1), ('Perceive…

### Analyze and visualize simulations

In [35]:
all_policies = all_sim_results + [simulation_random]
policy_names = [f"Policy {i}" for i in range(len(all_sim_results))] + ["Random Policy"]

for j in range(NUM_OBJECTIVES):
    print(f"\n--- {objectives[j]} ---")
    for idx, sim_res in enumerate(all_policies):
        raw_rewards = np.stack(sim_res['rewards'])
        rewards = raw_rewards.reshape(NUM_USERS, NUM_TIMESTEPS, NUM_OBJECTIVES)
        
        user_means = np.mean(rewards[:, :, j], axis=1) 
        grand_mean = np.mean(user_means)
        n = len(user_means)
        sem = stats.sem(user_means)
        ci = stats.t.interval(0.95, df=n-1, loc=grand_mean, scale=sem)
        
        print(f"{policy_names[idx]:<15} | Mean: {grand_mean:.4f} | 95% CI: ({ci[0]:.4f}, {ci[1]:.4f})")


--- Time Required ---
Policy 0        | Mean: 0.0399 | 95% CI: (0.0384, 0.0415)
Policy 1        | Mean: 0.0400 | 95% CI: (0.0385, 0.0415)
Policy 2        | Mean: 0.0391 | 95% CI: (0.0375, 0.0406)
Policy 3        | Mean: 0.0388 | 95% CI: (0.0372, 0.0403)
Policy 4        | Mean: 0.0399 | 95% CI: (0.0384, 0.0415)
Random Policy   | Mean: 0.0439 | 95% CI: (0.0423, 0.0455)

--- Fun ---
Policy 0        | Mean: 0.0413 | 95% CI: (0.0397, 0.0429)
Policy 1        | Mean: 0.0429 | 95% CI: (0.0412, 0.0446)
Policy 2        | Mean: 0.0414 | 95% CI: (0.0397, 0.0431)
Policy 3        | Mean: 0.0419 | 95% CI: (0.0402, 0.0436)
Policy 4        | Mean: 0.0423 | 95% CI: (0.0406, 0.0439)
Random Policy   | Mean: 0.0432 | 95% CI: (0.0417, 0.0448)

--- Perceived Usefulness ---
Policy 0        | Mean: 0.0661 | 95% CI: (0.0639, 0.0684)
Policy 1        | Mean: 0.0689 | 95% CI: (0.0665, 0.0713)
Policy 2        | Mean: 0.0654 | 95% CI: (0.0631, 0.0678)
Policy 3        | Mean: 0.0664 | 95% CI: (0.0640, 0.0689)
Policy

In [36]:
def plot_objective(objective_idx, is_cumulative):
    plt.figure(figsize=(10,6)) 
    obj_name = objectives[objective_idx]
    for idx, sim_res in enumerate(all_policies):
        name = policy_names[idx]
        raw_rewards = np.stack(sim_res['rewards'])
        rewards = raw_rewards.reshape(NUM_USERS, NUM_TIMESTEPS, NUM_OBJECTIVES)
        data = rewards[:, :, objective_idx]
        if is_cumulative:
            data = np.cumsum(data, axis=1)
        mean_rewards = np.mean(data, axis=0)
        style = '--' if "Random" in name else '-'
        plt.plot(mean_rewards, label=name, linestyle=style)

    plt.title(f"Average {obj_name} Over Time {'(Cumulative)' if is_cumulative else ''}")
    plt.xlabel("Time Step")
    plt.ylabel("Mean Reward")
    plt.legend()
    plt.grid(True)
    plt.show()

interact(
    plot_objective, 
    objective_idx=widgets.Dropdown(
        options=[(name, i) for i, name in enumerate(objectives)],
        value=0,
        description='Objective:',
    ),
    is_cumulative=widgets.Checkbox(
        value=True,
        description='Cumulative Sum',
    )
)

interactive(children=(Dropdown(description='Objective:', options=(('Time Required', 0), ('Fun', 1), ('Perceive…

<function __main__.plot_objective(objective_idx, is_cumulative)>

In [45]:
def build_idx_to_count(MAX_COUNT):
    dims = [MAX_COUNT + 1] * 4  # 4 categories
    grid = np.indices(dims)     # shape: (4, ..., ..., ..., ...)
    
    # reshape to (n_c, 4)
    idx_to_count = grid.reshape(4, -1).T
    
    return idx_to_count  # shape (n_c, 4)

def count_to_index(counts, MAX_COUNT):
    base = MAX_COUNT + 1
    return (
        counts[0] * base**3 +
        counts[1] * base**2 +
        counts[2] * base +
        counts[3]
    )

In [38]:
idx_to_count = build_idx_to_count(MAX_COUNT)

In [ ]:
def build_next_indices(idx_to_count, MAX_COUNT):
    n_c = idx_to_count.shape[0]
    base = MAX_COUNT + 1
    
    next_indices = np.zeros((4, n_c), dtype=int)  # 4 categories
    
    for k in range(4):
        c_next = idx_to_count.copy()
        
        # increment category k
        c_next[:, k] = np.minimum(c_next[:, k] + 1, MAX_COUNT)
        
        # normalize (your logic)
        min_vals = c_next.min(axis=1, keepdims=True)
        c_next = np.minimum(c_next - min_vals, MAX_COUNT)
        
        # map back to indices
        next_indices[k] = (
            c_next[:, 0] * base**3 +
            c_next[:, 1] * base**2 +
            c_next[:, 2] * base +
            c_next[:, 3]
        )
    
    return next_indices  # shape (4, n_c)

In [48]:
next_indices = build_next_indices(idx_to_count, MAX_COUNT)
print("Next indices shape:", next_indices.shape)  # Should be (4, n_c)

Next indices shape: (4, 256)


In [ ]:
def value_iteration_factored(env, weights, action_categories, gamma=0.9, theta=1e-6):
    n_u = 27
    n_c = (MAX_COUNT+1)**4
    V = np.zeros((n_u, n_c)) 
    
    # Pre-calculate scalarized rewards R[u, c, a]
    R = (env.reward_matrix * weights).sum(axis=2).reshape(n_u, n_c, env.nA)
    
    # Pre-extract completion probabilities p_c[u, c, a]
    # (Assuming the last objective is completion probability)
    P_comp = env.reward_matrix[:, :, -1].reshape(n_u, n_c, env.nA)

    while True:
        V_old = V.copy()
        Q = np.zeros((env.nA, n_u, n_c))
        
        for a in range(env.nA):
            # User state transition 
            # P_user_a is (n_u, n_u)
            P_user_a = env.transition_probs[:, a, :] 
            
            # Count state transition 
            # Probability of staying vs probability of incrementing
            p_c = P_comp[:, :, a] # Shape (n_u, n_c)

            k = action_categories[a]  # Category of the current action
            next_indices_k = next_indices[k]  # Shape (n_c,)
            
            # We calculate the expected future value for all user and count states at once
            # Values for staying in same count state
            V_next_stay = P_user_a @ V_old 
            # Values for incrementing count state
            V_next_increment = V_next_stay[:, next_indices_k]
            
            # Combine based on completion probability
            Q[a] = R[:, :, a] + gamma * ((1 - p_c) * V_next_stay + p_c * V_next_increment)

        V = np.max(Q, axis=0)
        
        if np.max(np.abs(V - V_old)) < theta:
            break
            
    return V.flatten(), np.argmax(Q, axis=0).flatten()

In [52]:
# use single-objective RL as baseline to compare
weights = np.array([0.166, 0.166, 0.166, 0.166, 0.166, 0.166])
env = mo_gym.make('user_env', num_actions=NUM_ACTIONS, 
                 num_objectives = NUM_OBJECTIVES, 
                 expert_score_matrix=expert_score_matrix, 
                 transition_probs=transition_probs, 
                 reward_matrix=reward_matrix, 
                 action_categories=action_categories,
                 MAX_COUNT=MAX_COUNT)

print(f"Reward matrix shape: {env.unwrapped.reward_matrix.shape}")
V, policy_vi = value_iteration_factored(env.unwrapped, weights=weights, action_categories=action_categories)

Reward matrix shape: (6912, 104, 6)


In [53]:
def simulate(env, num_users, policy=None, T=28):
    data_list = []
    for user in range(num_users):
        t = 0
        obs, _ = env.reset()  # later use initial state distribution
        done = False
        while not done and t < T:
            state_idx = utils.state_to_idx(tuple(obs), max_count=MAX_COUNT)
            action = policy[state_idx] if policy is not None else env.action_space.sample()
            obs_next, rewards, terminated, truncated, info = env.step(action)
            user_row = {
                'user': user,
                't': t,
                'action': action,
                'state': obs,
                'next_state': obs_next,
                'counts': obs[3:],
                'rewards': rewards
            }
            data_list.append(user_row)

            done = terminated or truncated
            obs = obs_next
            t += 1
    simulation_results = pd.DataFrame(data_list)
    return simulation_results

In [54]:
simulation_vi = simulate(env, 1000, policy_vi)


In [ ]:
all_policies = all_policies + [simulation_vi]
policy_names = policy_names + ["Policy VI (Equal Weights)"]
interact(
    plot_objective, 
    objective_idx=widgets.Dropdown(
        options=[(name, i) for i, name in enumerate(objectives)],
        value=0,
        description='Objective:',
    ),
    is_cumulative=widgets.Checkbox(
        value=True,
        description='Cumulative Sum',
    )
)

interactive(children=(Dropdown(description='Objective:', options=(('Time Required', 0), ('Fun', 1), ('Perceive…

<function __main__.plot_objective(objective_idx, is_cumulative)>

In [21]:
# use single-objective RL as baseline to compare
weights = [[1.0, 0, 0, 0, 0, 0], [0, 1.0, 0, 0, 0, 0], [0, 0, 1.0, 0, 0, 0], [0, 0, 0, 1.0, 0, 0], [0, 0, 0, 0, 1.0, 0], [0, 0, 0, 0, 0, 1.0]]
env = mo_gym.make('user_env', num_actions=NUM_ACTIONS, 
                 num_objectives = NUM_OBJECTIVES, 
                 expert_score_matrix=expert_score_matrix, 
                 transition_probs=transition_probs, 
                 reward_matrix=reward_matrix, 
                 action_categories=action_categories,
                 MAX_COUNT=MAX_COUNT)

all_single_policies = []
all_single_simulations = []
for w in weights:
    print(f"VI and simulation on weights: {w}")
    V, policy_vi = value_iteration_factored(env.unwrapped, weights=np.array(w))
    all_single_policies.append(policy_vi)
    print("Policy found, starting simulation")
    simulation_vi = simulate(env, 1000, policy_vi)
    all_single_simulations.append(simulation_vi)

all_single_simulations = all_single_simulations + [simulation_vi, simulation_random]
policy_names = [f"Policy {i}" for i in range(len(all_single_policies))] + ["Equal Weights", "Random Policy"]
all_policies = all_single_simulations
interact(
    plot_objective, 
    objective_idx=widgets.Dropdown(
        options=[(name, i) for i, name in enumerate(objectives)],
        value=0,
        description='Objective:',
    ),
    is_cumulative=widgets.Checkbox(
        value=True,
        description='Cumulative Sum',
    )
)

VI and simulation on weights: [1.0, 0, 0, 0, 0, 0]
Policy found, starting simulation
VI and simulation on weights: [0, 1.0, 0, 0, 0, 0]
Policy found, starting simulation
VI and simulation on weights: [0, 0, 1.0, 0, 0, 0]
Policy found, starting simulation
VI and simulation on weights: [0, 0, 0, 1.0, 0, 0]
Policy found, starting simulation
VI and simulation on weights: [0, 0, 0, 0, 1.0, 0]
Policy found, starting simulation
VI and simulation on weights: [0, 0, 0, 0, 0, 1.0]
Policy found, starting simulation


interactive(children=(Dropdown(description='Objective:', options=(('Time Required', 0), ('Fun', 1), ('Perceive…

<function __main__.plot_objective(objective_idx, is_cumulative)>